# 03 — Retrieval Metrics, By Hand

Notebooks 01 and 02 practiced judging *answers*. This one steps back one stage earlier in the pipeline: **did retrieval even find the right passage**, independent of whatever the model did with it afterward. Get this stage right and the rest of your evaluation stack has a chance. Get it wrong and nothing downstream can fix it -- a model can't answer from a passage it never received.

MS MARCO is a different dataset from the 20-example set in `eval_examples.json`, and deliberately so: it's the one dataset here that ships with **real, human-labeled relevance judgments** already attached, via an `is_selected` flag on each candidate passage. Nothing to label by hand -- only to rank yourself and check against the label.


In [ ]:
from datasets import load_dataset

marco = load_dataset("microsoft/ms_marco", "v2.1", split="validation", streaming=True)


## A real quirk, found immediately

The obvious move is "just take the first 5 queries." Try it, and most of them come back with **every** passage marked `is_selected: 0` -- MS MARCO includes plenty of queries where none of the 10 candidate passages was judged to actually answer it. That's a real, honest fact about retrieval-style benchmarks, not a bug in this notebook: sometimes there's genuinely no good passage in the candidate set, which is its own kind of retrieval failure, just not the kind this notebook is built to practice. Filter for queries that actually have at least one relevant passage, so Recall@K has something real to check against.


In [ ]:
def first_n_with_relevant_passage(dataset, n, scan_limit=2000):
    out = []
    for i, ex in enumerate(dataset):
        if sum(ex["passages"]["is_selected"]) >= 1:
            out.append(ex)
        if len(out) == n or i >= scan_limit:
            break
    return out


queries = first_n_with_relevant_passage(marco, 5)
for q in queries:
    relevant_idx = [i for i, s in enumerate(q["passages"]["is_selected"]) if s == 1]
    print(f"{q['query']!r} -> relevant at index(es) {relevant_idx}")


## Worked example 1 — one clearly relevant passage, several near-misses

Read the query, then read every candidate passage, and rank them yourself -- best match first -- **before** looking at which index MS MARCO actually marked relevant.


In [ ]:
q = queries[0]
print("Query:", q["query"])
print()
for i, text in enumerate(q["passages"]["passage_text"]):
    print(f"[{i}] {text[:160]}")


In [ ]:
relevant_idx = [i for i, s in enumerate(q["passages"]["is_selected"]) if s == 1]
print("Ground truth relevant index(es):", relevant_idx)


**Judgment:** the labeled passage, index 5 ("McDonald's Corporation is one of the most recognizable corporations... A corporation is a company or group of people authorized to act as a[n entity]..."), directly answers "what is a corporation" with a clean definition. Fair enough. But look at index 2 ("Corporation definition, an association of individuals, created by law...") and index 4 ("a government-owned corporation... engaged in a profit-making enterprise") -- both are *also* real, defensible answers to "what is a corporation," and neither is marked relevant.

This is the retrieval equivalent of what you found in notebook 02: **ground truth in these benchmarks is a human judgment call, not an objective fact.** MS MARCO's annotators picked one good passage and moved on; they weren't asked to mark every acceptable one. If your own retriever had surfaced index 2 instead of index 5, a naive Recall@1 check would call that a failure -- even though a human reading the actual passage would call it correct. Keep this in mind for every Recall@K number this notebook produces: it's measuring "did we match the one passage the annotator happened to pick," which is a reasonable proxy, not a perfect one.


In [ ]:
my_ranking_q0 = [5, 2, 4, 0, 6, 3, 9, 7, 1, 8]  # best match first, by my own reading above

recall_at_3 = any(idx in my_ranking_q0[:3] for idx in relevant_idx)
print("My Recall@3 for this query:", recall_at_3)


## Worked example 2 — more than one relevant passage

Query 2 has *two* passages marked relevant, not one. Recall@K just needs to catch at least one of them in the top K, but it's worth reading both to see why the annotator picked two instead of one.


In [ ]:
q = queries[1]
print("Query:", q["query"])
print()
for i, text in enumerate(q["passages"]["passage_text"]):
    print(f"[{i}] {text[:160]}")


In [ ]:
relevant_idx = [i for i, s in enumerate(q["passages"]["is_selected"]) if s == 1]
print("Ground truth relevant index(es):", relevant_idx)


**Judgment:** index 4 names the essay and states its subject; index 5 explains Carson's actual argument (pesticides causing more harm than good). Together they answer "why did she write it" more completely than either alone -- naming the topic isn't the same as stating the argument. This is worth noticing: Recall@K as commonly defined only asks "did at least one relevant passage make the cut." It doesn't check whether you got *both*, even when getting both would make for a meaningfully better final answer. That's a real limitation of the metric, not just this notebook's version of it.


In [ ]:
my_ranking_q1 = [5, 4, 7, 6, 2, 3, 9, 0, 1, 8]

recall_at_3 = any(idx in my_ranking_q1[:3] for idx in relevant_idx)
print("My Recall@3 for this query:", recall_at_3)


## Your turn: the remaining 3 queries

Same process. Read all 10 candidates, rank them yourself, then check against the label.


In [ ]:
for q in queries[2:]:
    print("Query:", q["query"])
    for i, text in enumerate(q["passages"]["passage_text"]):
        print(f"  [{i}] {text[:150]}")
    relevant_idx = [i for i, s in enumerate(q["passages"]["is_selected"]) if s == 1]
    print("  Ground truth relevant index(es):", relevant_idx)
    print()


In [ ]:
# Fill in your own ranking for each, best match first:
my_rankings = {
    0: my_ranking_q0,
    1: my_ranking_q1,
    2: [],  # queries[2] -- "symptoms of a dying mouse"
    3: [],  # queries[3] -- "average number of lightning strikes per day"
    4: [],  # queries[4] -- "can you burn your lawn with fertilizer"
}


## From manual ranking to code

Once you've ranked all 5 by hand, computing Recall@K in code is nothing more than automating the exact comparison you just did four times.


In [ ]:
def recall_at_k(ranking: list[int], relevant_idx: list[int], k: int) -> bool:
    return any(idx in ranking[:k] for idx in relevant_idx)


results = []
for i, q in enumerate(queries):
    relevant_idx = [j for j, s in enumerate(q["passages"]["is_selected"]) if s == 1]
    ranking = my_rankings.get(i, [])
    if not ranking:
        continue
    results.append({"query": q["query"], "recall_at_3": recall_at_k(ranking, relevant_idx, 3)})

for r in results:
    print(f"{r['query']!r:60} -> Recall@3: {r['recall_at_3']}")


You just computed Recall@3 by hand, on real ground truth, before a metrics library ever entered the picture. Every retrieval-evaluation framework you'll meet later in this series is running exactly this check -- "did the relevant item make it into the top K" -- just automated and averaged across thousands of queries instead of 5.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Recall@K | Did at least one relevant item make it into the top K results |
| Candidate passages | The full set retrieval returned, before any ranking or filtering |
| Ground truth relevance | A human's judgment of "this passage answers the query" -- a call, not a fact |

The mechanical part of Recall@K is genuinely simple -- you just did it by hand. The harder, more useful skill is what worked example 1 forced: reading a "relevant" label skeptically instead of treating it as gospel, because the annotator who created it was making the same kind of judgment call you make in notebooks 01 and 02.

**Next up:** notebook `04` asks a question Recall@K can't answer at all -- retrieval found the right *document*, but was what actually reached the model any good?

**Exercises:**

- Finish ranking `my_rankings` for the 3 remaining queries and re-run the Recall@3 check.
- Compute Recall@1 (not just Recall@3) for all 5 queries using your own rankings -- how many still pass?
- For query 0, would a retriever that returned passages [2, 4, 5] in some order, but never the exact top-3 slot MS MARCO expects, still deserve credit for a "good" retrieval in your judgment? Write down why or why not.
